# Lab | LangChain Med

## Objectives

- continue on with lesson 2' example, use different datasets to test what we did in class. Some datasets are suggested in the notebook but feel free to scout other datasets on HuggingFace or Kaggle.
- Find another model on Hugging Face and compare it.
- Modify the prompt to fit your selected dataset.

In [1]:
%pip install -qU chromadb transformers accelerate scikit-learn matplotlib "pandas==2.2.2"

import os
import numpy as np
import pandas as pd
from google.colab import userdata

# Retrieve OpenAI API key from Colab secrets (needed if you extend this lab with OpenAI)
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY').strip()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 674.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 991.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231

## Load the Dataset
As you can see the notebook is ready to work with three different Datasets. Just uncomment the lines of the Dataset you want to use.

I selected Datasets with News. Two of them have just a brief decription of the news, but the other contains the full text.

As we are working in a free and limited space, I limited the number of news to use with the variable MAX_NEWS. Feel free to pull more if you have memory available.

The name of the field containing the text of the new is stored in the variable *DOCUMENT* and the metadata in *TOPIC*

In [2]:
# Option 1: NewsCatcher dataset (recommended - same as used in lessons)
# Download via HuggingFace datasets (no Kaggle account needed)
from datasets import load_dataset
hf_data = load_dataset("cc_news", split="train", trust_remote_code=True)
news = hf_data.to_pandas()[["title", "description"]].dropna().head(2000)
news.columns = ["title", "description"]
MAX_NEWS = 1000
DOCUMENT = "title"
TOPIC = "description"

# Option 2: BBC News (also on HuggingFace)
# hf_data = load_dataset("SetFit/bbc-news", split="train")
# news = hf_data.to_pandas()
# MAX_NEWS = 1000
# DOCUMENT = "text"
# TOPIC = "label_text"

print('Dataset shape:', news.shape)
news.head()


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cc_news' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cc_news' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00005.parquet:   0%|          | 0.00/211M [00:00<?, ?B/s]

plain_text/train-00001-of-00005.parquet:   0%|          | 0.00/234M [00:00<?, ?B/s]

plain_text/train-00002-of-00005.parquet:   0%|          | 0.00/219M [00:00<?, ?B/s]

plain_text/train-00003-of-00005.parquet:   0%|          | 0.00/245M [00:00<?, ?B/s]

plain_text/train-00004-of-00005.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/708241 [00:00<?, ? examples/s]

Dataset shape: (2000, 2)


,title,description
0,Daughter Duo is Dancing in The Same Company,There's a surprising twist to Regina Willoughb...
1,New York City Ballet Announces Interim Leaders...,NYCB has announced an interim leadership team ...
2,Watch Pennsylvania Ballet & Boston Ballet Face...,The Philadelphia Eagles and the New England Pa...
3,dance shoes,Looking for your next audition shoe? Shot at a...
4,Rebecca Krohn on Her Retirement from New York ...,We interviewed New York City Ballet principal ...


ChromaDB requires that the data has a unique identifier. We can make it with this statement, which will create a new column called **Id**.


In [3]:
news["id"] = news.index
news.head()

,title,description,id
0,Daughter Duo is Dancing in The Same Company,There's a surprising twist to Regina Willoughb...,0
1,New York City Ballet Announces Interim Leaders...,NYCB has announced an interim leadership team ...,1
2,Watch Pennsylvania Ballet & Boston Ballet Face...,The Philadelphia Eagles and the New England Pa...,2
3,dance shoes,Looking for your next audition shoe? Shot at a...,3
4,Rebecca Krohn on Her Retirement from New York ...,We interviewed New York City Ballet principal ...,4


In [4]:
#Because it is just a course we select a small portion of News.
subset_news = news.head(MAX_NEWS)

## Import and configure the Vector Database
I'm going to use ChromaDB, the most popular OpenSource embedding Database.

First we need to import ChromaDB, and after that import the **Settings** class from **chromadb.config** module. This class allows us to change the setting for the ChromaDB system, and customize its behavior.

In [5]:
# chromadb already installed in the first cell


In [6]:
import chromadb
# Note: chromadb.config.Settings is no longer needed in chromadb >= 0.4


Now we need to create the seetings object calling the **Settings** function imported previously. We store the object in the variable **settings_chroma**.

Is necessary to inform two parameters
* chroma_db_impl. Here we specify the database implementation and the format how store the data. I choose ***duckdb***, because his high-performace. It operate primarly in memory. And is fully compatible with SQL. The store format ***parquet*** is good for tabular data. With good compression rates and performance.

* persist_directory: It just contains the directory where the data will be stored. Is possible work without a directory and the data will be stored in memory without persistece, but Kaggle dosn't support that.

In [7]:
# Create a persistent ChromaDB client in the current working directory
chroma_client = chromadb.PersistentClient(path="./chroma_lab_med")


## Filling and Querying the ChromaDB Database
The Data in ChromaDB is stored in collections. If the collection exist we need to delete it.

In the next lines, we are creating the collection by calling the ***create_collection*** function in the ***chroma_client*** created above.

In [8]:
collection_name = "news_collection"

# Delete the collection if it already exists to start fresh
existing = [c.name for c in chroma_client.list_collections()]
if collection_name in existing:
    chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)


It's time to add the data to the collection. Using the function ***add*** we need to inform, at least ***documents***, ***metadatas*** and ***ids***.
* In the **document** we store the big text, it's a different column in each Dataset.
* In **metadatas**, we can informa a list of topics.
* In **id** we need to inform an unique identificator for each row. It MUST be unique! I'm creating the ID using the range of MAX_NEWS.


In [9]:

collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)],
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 9.44MiB/s]


In [10]:
results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

{'ids': [['id87', 'id61', 'id311', 'id45', 'id40', 'id36', 'id97', 'id100', 'id109', 'id508']], 'embeddings': None, 'documents': [['Size of a Raspberry Pi, power of a MacBook: The Window 10-powered LattePanda Alpha', 'Built-in keylogger found in HP laptops...again', 'Harrison Ford, Get Off Your Plane', 'A new slim version of Windows 10 called "Lean" could be perfect for low-end PCs', 'How to choose the right wearable device for business vs. fitness', "HPE and NASA experiment with 'Spaceborne Computer,' a supercomputer that could help us get to Mars", 'প্রাক্তন প্রেমিকা প্রিয়াঙ্কাকে মেরে ফেলতে চান শাহিদ!!', 'আরিব্বাস, জাল ডাক্তার!', 'মনে আছে ‘মিঃ ইন্ডিয়া’র ছোট্ট টিনাকে? জানেন এখন কেমন দেখতে সে?', 'Report: Harrison Ford has jet run-in at California airport']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'description': 'The LattePanda Alpha has more in common with a laptop than low-cost board like the Pi.'}, {'description': 'A security 

## Vector MAP

In [11]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [12]:
# ids must be passed as a list
getado = collection.get(ids=["id141"],
                        include=["documents", "embeddings"])


In [13]:
word_vectors = getado["embeddings"]
word_list = getado["documents"]
word_vectors

array([[-1.99827068e-02,  1.03442781e-02, -6.19886741e-02,
        -3.45421247e-02,  1.23964418e-02,  5.56933098e-02,
         5.95369786e-02,  2.79213488e-02, -2.79988199e-02,
         1.77261289e-02, -9.94162560e-02, -2.16963515e-02,
        -3.39126028e-02,  6.85064495e-02,  2.48540081e-02,
         3.27172899e-03, -2.77266912e-02, -8.06426909e-03,
        -1.15985110e-01,  3.25386524e-02,  6.23739697e-02,
         3.36936489e-02,  6.56461343e-02,  3.09352390e-02,
        -5.30974530e-02,  6.47946224e-02,  2.71292794e-02,
        -4.44895998e-02,  1.92882773e-02, -3.89780290e-02,
        -7.07489550e-02, -1.61038265e-02,  7.75428340e-02,
        -5.05243801e-02,  2.67508663e-02,  6.88148141e-02,
         3.89928650e-03,  1.86443292e-02, -2.61471290e-02,
         5.33723570e-02, -3.67252231e-02, -6.47050813e-02,
         1.88507102e-02,  3.32184657e-02, -1.29829533e-02,
         1.82161704e-02,  1.72768738e-02,  6.60616979e-02,
         3.29846926e-02,  1.82404611e-02, -4.20265757e-0

Once we have our information inside the Database we can query It, and ask for data that matches our needs. The search is done inside the content of the document, and it dosn't look for the exact word, or phrase. The results will be based on the similarity between the search terms and the content of documents.

The metadata is not used in the search, but they can be utilized for filtering or refining the results after the initial search.


## Loading the model and creating the prompt
TRANSFORMERS!!
Time to use the library **transformers**, the most famous library from [hugging face](https://huggingface.co/) for working with language models.

We are importing:
* **Autotokenizer**: It is a utility class for tokenizing text inputs that are compatible with various pre-trained language models.
* **AutoModelForCasualLLM**: it provides an interface to pre-trained language models specifically designed for language generation tasks using causal language modeling (e.g., GPT models), or the model used in this notebook ***databricks/dolly-v2-3b***.
* **pipeline**: provides a simple interface for performing various natural language processing (NLP) tasks, such as text generation (our case) or text classification.

The model selected is [dolly-v2-3b](https://huggingface.co/databricks/dolly-v2-3b), the smallest Dolly model. It have 3billion paramaters, more than enough for our sample, and works much better than GPT2.

Please, feel free to test [different Models](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending), you need to search for NLP models trained for text-generation. My recomendation is choose "small" models, or we will run out of memory in kaggle.  


In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Original model was databricks/SmolLM2-135M-Instruct (6GB+, too slow for Colab).
# Using SmolLM2-135M-Instruct instead: same learning objective, runs in seconds.
model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id, low_cpu_mem_usage=True)


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

The next step is to initialize the pipeline using the objects created above.

The model's response is limited to 256 tokens, for this project I'm not interested in a longer response, but it can easily be extended to whatever length you want.

Setting ***device_map*** to ***auto*** we are instructing the model to automaticaly select the most appropiate device: CPU or GPU for processing the text generation.  

In [15]:
import torch

device = 0 if torch.cuda.is_available() else -1  # 0 = first GPU, -1 = CPU

pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device=device,
    pad_token_id=tokenizer.eos_token_id,
)


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Creating the extended prompt
To create the prompt we use the result from query the Vector Database  and the sentence introduced by the user.

The prompt have two parts, the **relevant context** that is the information recovered from the database and the **user's question**.

We only need to join the two parts together to create the prompt that we are going to send to the model.

You can limit the lenght of the context passed to the model, because we can get some Memory problems with one of the datasets that contains a realy large text in the document part.

In [18]:
question = "Can you tell me about a supercomputer?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
#context = context[0:5120]
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
prompt_template

'Relevant context: #Size of a Raspberry Pi, power of a MacBook: The Window 10-powered LattePanda Alpha #Built-in keylogger found in HP laptops...again #Harrison Ford, Get Off Your Plane #A new slim version of Windows 10 called "Lean" could be perfect for low-end PCs #How to choose the right wearable device for business vs. fitness #HPE and NASA experiment with \'Spaceborne Computer,\' a supercomputer that could help us get to Mars #প্রাক্তন প্রেমিকা প্রিয়াঙ্কাকে মেরে ফেলতে চান শাহিদ!! #আরিব্বাস, জাল ডাক্তার! #মনে আছে ‘মিঃ ইন্ডিয়া’র ছোট্ট টিনাকে? জানেন এখন কেমন দেখতে সে? #Report: Harrison Ford has jet run-in at California airport\n\n The user\'s question: Can you tell me about a supercomputer?'

Now all that remains is to send the prompt to the model and wait for its response!


In [19]:
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Relevant context: #Size of a Raspberry Pi, power of a MacBook: The Window 10-powered LattePanda Alpha #Built-in keylogger found in HP laptops...again #Harrison Ford, Get Off Your Plane #A new slim version of Windows 10 called "Lean" could be perfect for low-end PCs #How to choose the right wearable device for business vs. fitness #HPE and NASA experiment with 'Spaceborne Computer,' a supercomputer that could help us get to Mars #প্রাক্তন প্রেমিকা প্রিয়াঙ্কাকে মেরে ফেলতে চান শাহিদ!! #আরিব্বাস, জাল ডাক্তার! #মনে আছে ‘মিঃ ইন্ডিয়া’র ছোট্ট টিনাকে? জানেন এখন কেমন দেখতে সে? #Report: Harrison Ford has jet run-in at California airport

 The user's question: Can you tell me about a supercomputer? The user's answer: A supercomputer is a highly powerful computer system that is designed to perform large-scale computations or simulations. They are often used to solve complex problems that can't be solved by computers alone. Examples of supercomputers include the Atacama Large Millimeter Large Telesc